In [178]:
import joblib
import pandas as pd
import matplotlib.pyplot as plt
from sklearn import set_config
from tempfile import TemporaryDirectory
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import FunctionTransformer
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import AdaBoostClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.ensemble import VotingClassifier
from sklearn.ensemble import StackingClassifier
import xgboost as xgb
#from lightgbm import LGBMClassifier
#from catboost import CatBoostClassifier
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.model_selection import RandomizedSearchCV
from sklearn.model_selection import cross_val_score
from sklearn.metrics import accuracy_score
from sklearn.metrics import balanced_accuracy_score
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix
from sklearn import metrics
import seaborn as sns
import numpy as np
import warnings

In [179]:
target_column = "health_condition"
df = pd.read_csv("data/train.csv")

In [180]:
df.drop('id', axis=1, inplace=True)

df[target_column].replace({'unhealthy':0, 'at-risk':1, 'fit':2}).astype('int')

df['diet_type'] = df['diet_type'].astype('category')
df['gender'] = df['gender'].astype('category')
df['stress_level'] = df['stress_level'].replace({'low':0, 'medium':1, 'high':2})
df['sleep_quality'] = df['sleep_quality'].replace({'poor':0, 'average':1, 'good':2})
df['physical_activity_level'] = df['physical_activity_level'].replace({'sedentary':0, 'moderate':1, 'active':2})
df['smoking_alcohol'] = df['smoking_alcohol'].replace({'no':0, 'occasional':1, 'yes':2})

object_cols = ['stress_level', 'sleep_quality', 'physical_activity_level', 'smoking_alcohol']
for col in object_cols:
    df[col] = df[col].astype(np.float64)

In [181]:
df['calorie_expenditure_per_step'] = df['calorie_expenditure'] / (df['step_count']+1)
df['step_speed'] = df['step_count'] / (df['exercise_duration']+1)
df['calorie_expenditure_per_min'] = df['calorie_expenditure'] / (df['exercise_duration']+1)
df['calorie_expenditure_per_bmi'] = df['calorie_expenditure'] / (df['bmi']+1)

df.info()

<class 'pandas.DataFrame'>
RangeIndex: 690088 entries, 0 to 690087
Data columns (total 18 columns):
 #   Column                        Non-Null Count   Dtype   
---  ------                        --------------   -----   
 0   health_condition              690088 non-null  str     
 1   sleep_duration                614089 non-null  float64 
 2   heart_rate                    682255 non-null  float64 
 3   bmi                           676190 non-null  float64 
 4   calorie_expenditure           637235 non-null  float64 
 5   step_count                    676172 non-null  float64 
 6   exercise_duration             683187 non-null  float64 
 7   water_intake                  646611 non-null  float64 
 8   diet_type                     683187 non-null  category
 9   stress_level                  607277 non-null  float64 
 10  sleep_quality                 631757 non-null  float64 
 11  physical_activity_level       653467 non-null  float64 
 12  smoking_alcohol               661506 non-

In [182]:
df.to_csv('data/intermediate/data_features.csv', index=False)

In [183]:
y = df[target_column].replace({'unhealthy':0, 'at-risk':1, 'fit':2}).astype('int')
X = df.drop(target_column, axis=1)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [184]:
X_train.info()

<class 'pandas.DataFrame'>
Index: 552070 entries, 148924 to 121958
Data columns (total 17 columns):
 #   Column                        Non-Null Count   Dtype   
---  ------                        --------------   -----   
 0   sleep_duration                491266 non-null  float64 
 1   heart_rate                    545802 non-null  float64 
 2   bmi                           541053 non-null  float64 
 3   calorie_expenditure           509705 non-null  float64 
 4   step_count                    540909 non-null  float64 
 5   exercise_duration             546550 non-null  float64 
 6   water_intake                  517224 non-null  float64 
 7   diet_type                     546580 non-null  category
 8   stress_level                  485727 non-null  float64 
 9   sleep_quality                 505469 non-null  float64 
 10  physical_activity_level       522744 non-null  float64 
 11  smoking_alcohol               529184 non-null  float64 
 12  gender                        535022 non-

In [185]:
missing_num_features = ["step_count", "bmi", "heart_rate", "exercise_duration", "physical_activity_level", "smoking_alcohol", "water_intake"]
missing_cat_features = ["gender", "diet_type", "sleep_quality", "stress_level"]

for ft in missing_num_features:
    me = X_train[ft].mean()

    X_train[ft] = X_train[ft].fillna(me)
    X_test[ft] = X_test[ft].fillna(me)

for ft in missing_cat_features:
    mode = X_train[ft].mode().iloc[0]

    X_train[ft] = X_train[ft].fillna(mode)
    X_test[ft] = X_test[ft].fillna(mode)

In [186]:
def group_numerical_impute(frame, impute_col, group_col, bins=8):
    frame[f"{group_col}_binned"] = pd.cut(frame[group_col], bins=bins)
    frame[impute_col] = frame.groupby(by=[f"{group_col}_binned"])[impute_col].transform(lambda x: x.fillna(x.mean()))
    frame.drop(f"{group_col}_binned", axis=1, inplace=True)

In [187]:
group_numerical_impute(X_train, "calorie_expenditure", "exercise_duration")
group_numerical_impute(X_test, "calorie_expenditure", "exercise_duration")

group_numerical_impute(X_train, "sleep_duration", "stress_level")
group_numerical_impute(X_test, "sleep_duration", "stress_level")

X_train["sleep_duration"] = X_train.groupby(by=[f"sleep_quality"])["sleep_duration"].transform(lambda x: x.fillna(x.mean()))
X_test["sleep_duration"] = X_test.groupby(by=[f"sleep_quality"])["sleep_duration"].transform(lambda x: x.fillna(x.mean()))

In [188]:
X_train['calorie_expenditure_per_step'] = X_train['calorie_expenditure'] / (X_train['step_count']+1)
X_train['step_speed'] = X_train['step_count'] / (X_train['exercise_duration']+1)
X_train['calorie_expenditure_per_min'] = X_train['calorie_expenditure'] / (X_train['exercise_duration']+1)
X_train['calorie_expenditure_per_bmi'] = X_train['calorie_expenditure'] / (X_train['bmi']+1)

X_test['calorie_expenditure_per_step'] = X_test['calorie_expenditure'] / (X_test['step_count']+1)
X_test['step_speed'] = X_test['step_count'] / (X_test['exercise_duration']+1)
X_test['calorie_expenditure_per_min'] = X_test['calorie_expenditure'] / (X_test['exercise_duration']+1)
X_test['calorie_expenditure_per_bmi'] = X_test['calorie_expenditure'] / (X_test['bmi']+1)

In [189]:
X_train.info()

<class 'pandas.DataFrame'>
Index: 552070 entries, 148924 to 121958
Data columns (total 17 columns):
 #   Column                        Non-Null Count   Dtype   
---  ------                        --------------   -----   
 0   sleep_duration                552070 non-null  float64 
 1   heart_rate                    552070 non-null  float64 
 2   bmi                           552070 non-null  float64 
 3   calorie_expenditure           552070 non-null  float64 
 4   step_count                    552070 non-null  float64 
 5   exercise_duration             552070 non-null  float64 
 6   water_intake                  552070 non-null  float64 
 7   diet_type                     552070 non-null  category
 8   stress_level                  552070 non-null  float64 
 9   sleep_quality                 552070 non-null  float64 
 10  physical_activity_level       552070 non-null  float64 
 11  smoking_alcohol               552070 non-null  float64 
 12  gender                        552070 non-

In [190]:
X_train.to_csv('data/intermediate/train_features_imputed.csv', index=False)
X_test.to_csv('data/intermediate/test_features_imputed.csv', index=False)